# Large detector planes: full-track generation check

Generate events only, with no Hamiltonian construction, reconstruction, or QSVT filtering. Each plane is 200 ? 200 mm and adjacent planes are 33 mm apart. Check that every generated track produces exactly one hit on every plane, for 5?10 layers and all 30 event seeds.

For straight tracks from the origin, the largest transverse displacement at layer 10 is approximately `397 * tan(0.2) = 80.48 mm`, below the 100 mm half-width. Scattering and measurement errors are retained; the checks below validate the actual generated events. Finite planes cannot guarantee containment for every possible Gaussian noise draw.


In [15]:
from pathlib import Path
import sys
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src/lhcb_velo_toy').is_dir()
            or (p / 'LHCb_VeLo_Toy_Model/src/lhcb_velo_toy').is_dir())
if not (ROOT / 'src/lhcb_velo_toy').is_dir():
    ROOT = ROOT / 'LHCb_VeLo_Toy_Model'
sys.path.insert(0, str(ROOT / 'src'))
from lhcb_velo_toy.generation import PlaneGeometry, StateEventGenerator

plt.rcParams['figure.dpi'] = 110
print('Using', ROOT)


Using c:\Users\joseh\BTR\LHCb_VeLo_Toy_Model


The minimal required detector planes should be 161mm for 10 layers. to have a bigger margin the detector planes in these simulated events will have the size 200mm (100mm Lx and 100 mm Ly)

In [16]:
N_LAYERS = 10             # selected event shown in the plots
LAYER_SCAN = [5, 6, 7, 8, 9, 10]
N_TRACKS = 400
PLANE_HALF_X = 100.0      # mm: full width = 200 mm
PLANE_HALF_Y = 100.0      # mm: full height = 200 mm
DZ = 33.0                # mm
Z_FIRST = 100.0          # mm
MEASUREMENT_ERROR = 0.01 # mm
COLLISION_NOISE = 1e-4
PHI_MIN, PHI_MAX = -0.2, 0.2
THETA_MIN, THETA_MAX = -0.2, 0.2
SEED = 202
EVENT_SEEDS = tuple(range(1, 31))

assert N_TRACKS > 0 and PLANE_HALF_X > 0 and PLANE_HALF_Y > 0 and DZ > 0
assert all(isinstance(n, int) and n >= 2 for n in [N_LAYERS, *LAYER_SCAN])
assert -np.pi/2 < PHI_MIN < PHI_MAX < np.pi/2
assert -np.pi/2 < THETA_MIN < THETA_MAX < np.pi/2


In [17]:
def make_event(n_layers, seed=SEED):
    geometry = PlaneGeometry(
        module_id=list(range(n_layers)), lx=[PLANE_HALF_X] * n_layers,
        ly=[PLANE_HALF_Y] * n_layers,
        z=[Z_FIRST + i * DZ for i in range(n_layers)])
    gen = StateEventGenerator(
        detector_geometry=geometry, events=1, n_particles=[N_TRACKS],
        measurement_error=MEASUREMENT_ERROR, collision_noise=COLLISION_NOISE,
        phi_min=PHI_MIN, phi_max=PHI_MAX,
        theta_min=THETA_MIN, theta_max=THETA_MAX, seed=seed)
    gen.generate_random_primary_vertices({'x': 0, 'y': 0, 'z': 0})
    gen.generate_particles([[{'type': 'pion', 'mass': 139.6, 'q': 1}] * N_TRACKS])
    gen.generate_complete_events()
    return gen


def check_event(gen, n_layers, seed):
    event = gen.true_event
    context = f'{n_layers} layers, seed {seed}'
    assert len(event.tracks) == N_TRACKS, context
    assert len(event.modules) == n_layers, context
    hits_by_id = {h.hit_id: h for h in event.hits}
    assert len(hits_by_id) == len(event.hits), context
    modules = {m.module_id: m for m in event.modules}
    expected_modules = set(range(n_layers))
    counts = []
    for track in event.tracks:
        hits = [hits_by_id[i] for i in track.hit_ids]
        counts.append(len(hits))
        assert len(hits) == n_layers, f'{context}: track {track.track_id} has {len(hits)} hits'
        assert {h.module_id for h in hits} == expected_modules, context
        assert all(h.track_id == track.track_id for h in hits), context
    assert len(event.hits) == N_TRACKS * n_layers, context
    per_layer = Counter(h.module_id for h in event.hits)
    assert all(per_layer[m] == N_TRACKS for m in expected_modules), context
    margins = []
    for hit in event.hits:
        module = modules[hit.module_id]
        assert hit.z == module.z, context
        margin = min(module.lx - abs(hit.x), module.ly - abs(hit.y))
        assert np.isfinite([hit.x, hit.y, hit.z]).all() and margin >= 0, context
        margins.append(margin)
    return dict(layers=n_layers, seed=seed, tracks=len(event.tracks),
                full_tracks=sum(c == n_layers for c in counts), hits=len(event.hits),
                min_hits=min(counts), max_hits=max(counts), margin_mm=min(margins))


In [18]:
# Every seed must pass every assertion; failures identify the layer count and seed.
scan_results = []
for n_layers in LAYER_SCAN:
    for seed in EVENT_SEEDS:
        scan_results.append(check_event(make_event(n_layers, seed), n_layers, seed))
    print(f'{n_layers} layers: all {len(EVENT_SEEDS)} events passed')

rows = ['| Layers | Events | Full tracks / generated | Total hits | Hits per track | Smallest edge margin (mm) |',
        '|---|---|---|---|---|---|']
for n_layers in LAYER_SCAN:
    group = [r for r in scan_results if r['layers'] == n_layers]
    rows.append(f"| {n_layers} | {len(group)} | {sum(r['full_tracks'] for r in group)} / {sum(r['tracks'] for r in group)} | {sum(r['hits'] for r in group)} | {min(r['min_hits'] for r in group)}?{max(r['max_hits'] for r in group)} | {min(r['margin_mm'] for r in group):.3f} |")
display(Markdown('\n'.join(rows)))


5 layers: all 30 events passed
6 layers: all 30 events passed
7 layers: all 30 events passed
8 layers: all 30 events passed
9 layers: all 30 events passed
10 layers: all 30 events passed


| Layers | Events | Full tracks / generated | Total hits | Hits per track | Smallest edge margin (mm) |
|---|---|---|---|---|---|
| 5 | 30 | 12000 / 12000 | 60000 | 5?5 | 52.933 |
| 6 | 30 | 12000 / 12000 | 72000 | 6?6 | 46.235 |
| 7 | 30 | 12000 / 12000 | 84000 | 7?7 | 39.550 |
| 8 | 30 | 12000 / 12000 | 96000 | 8?8 | 32.831 |
| 9 | 30 | 12000 / 12000 | 108000 | 9?9 | 26.179 |
| 10 | 30 | 12000 / 12000 | 120000 | 10?10 | 19.456 |